# 02. XBRL 본문 파싱 → 사실(Fact) DB 구축

다운로드된 XBRL 본문을 파싱하여 (회사·기간·계정·단위·연결/별도·값) 6요소의 사실 레코드로 변환합니다.

**소요 시간**: 약 1-2분

## 2.1 환경 준비

In [ ]:
import os, sys
sys.path.insert(0, '/content/repo')

# 입출력 경로
RAW_DIR = '/content/data/raw'
PROCESSED_DIR = '/content/data/processed'
DB_DIR = '/content/db'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

## 2.2 XBRL 파싱

XBRL 본문 → JSON Lines 형식으로 변환:
- `facts.jsonl`: 정형 사실 데이터 (수치)
- `sections.jsonl`: 본문 섹션 (서술 텍스트)

In [ ]:
# preprocessor_v2.py 실행
from src.data.preprocessor_v2 import process_all_companies

stats = process_all_companies(
    input_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
)

print(f"\n처리 결과:")
print(f"  사실 레코드: {stats['facts']:,}건")
print(f"  본문 섹션: {stats['sections']:,}개")
print(f"  처리 파일: {stats['files']}개")

## 2.3 사실 DB 구축 (SQLite)

JSON Lines 형식의 사실을 SQLite로 색인 → 평균 30ms 이내 조회 가능

In [ ]:
from src.data.build_fact_db import build_fact_db

build_fact_db(
    input_path=f'{PROCESSED_DIR}/facts.jsonl',
    output_path=f'{DB_DIR}/facts.db',
)

# 결과 확인
import sqlite3
conn = sqlite3.connect(f'{DB_DIR}/facts.db')
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM facts")
total = cursor.fetchone()[0]

cursor.execute("SELECT corp_name, COUNT(*) FROM facts GROUP BY corp_name")
print(f"\n사실 DB 구축 완료:")
print(f"  총 레코드: {total:,}건")
print(f"\n  기업별 분포:")
for name, count in cursor.fetchall():
    print(f"    {name}: {count:,}건")

conn.close()

## 2.4 샘플 조회 테스트

In [ ]:
import sqlite3
conn = sqlite3.connect(f'{DB_DIR}/facts.db')
cursor = conn.cursor()

# 삼성전자 2025년 매출 조회
query = '''
SELECT corp_name, period, account_name, value, unit
FROM facts
WHERE corp_name = '삼성전자'
  AND account_name LIKE '%매출%'
  AND period LIKE '%2025%'
LIMIT 5
'''
cursor.execute(query)
print("샘플 조회 (삼성전자 2025년 매출):")
print(f"{'기업':<10} {'기간':<15} {'계정':<25} {'값':>20} {'단위':<6}")
print("-" * 80)
for row in cursor.fetchall():
    print(f"{row[0]:<10} {row[1]:<15} {row[2]:<25} {row[3]:>20,} {row[4]:<6}")

conn.close()

## 2.5 다음 단계

본문 섹션을 RAG 색인으로 변환합니다.

→ **03_rag_indexing.ipynb** 로 진행